# Python for AI — Class 12
### Exception handling and file handling

Until now, one bad input crashed your whole program. Today you learn to **catch** errors and decide what happens next, and to **raise** your own when data is wrong. Then you step outside the notebook: **reading and writing files** — plain text, CSV and JSON — so data survives after the program stops. Every AI project you build later starts by loading a file and ends by saving one.

**How to use this notebook:** run each cell with the play button, or `Shift + Enter`.

Cells marked **BREAKS ON PURPOSE** are *supposed* to show a red error. Cells marked **WRONG ON PURPOSE** run fine and give the *wrong answer* — those are the dangerous ones. Don't fix either before class; that is the lesson.

# 1. Why exceptions? A crash stops everything

On Day 2 you learned to *read* a traceback. Now look at what a crash actually costs: one bad value, and every line after it never runs.

In [1]:
# BREAKS ON PURPOSE - one bad value kills the whole loop
prices = ["1200", "850", "free", "400"]

total = 0
for p in prices:
    total += int(p)          # int("free") -> ValueError
    print("running total:", total)

print("Total:", total)       # never reached

running total: 1200
running total: 2050


ValueError: invalid literal for int() with base 10: 'free'

**ValueError: invalid literal for int() with base 10: 'free'.** The first two prices were fine, but the program died on the third and the fourth was never looked at. In a real script processing 10,000 rows, one bad row would throw away all the work.

# 2. `try` / `except` — catch the error, keep going

Put the risky line inside `try`. If it raises an error, Python jumps to `except` instead of crashing:

```
try:
    risky line          ← if this fails...
except ValueError:
    what to do instead  ← ...run this, then carry on
```

In [2]:
prices = ["1200", "850", "free", "400"]

total = 0
for p in prices:
    try:
        total += int(p)
    except ValueError:
        print(f"Skipping bad price: {p!r}")

print("Total:", total)

Skipping bad price: 'free'
Total: 2450


Same data, but now the bad row is reported and skipped, and the other three are counted. That's the whole idea: **you** decide what a failure means, not Python.

## Common built-in exceptions

You'll meet these constantly. Learn to recognise the name — it tells you what went wrong before you read anything else.

| Exception | Typical cause |
|-----------|---------------|
| `ValueError` | right type, wrong value — `int("abc")` |
| `TypeError` | wrong type — `"5" + 5` |
| `ZeroDivisionError` | `10 / 0` |
| `IndexError` | list position that doesn't exist — `[1, 2][5]` |
| `KeyError` | dict key that doesn't exist — `{"a": 1}["b"]` |
| `FileNotFoundError` | opening a file that isn't there |
| `AttributeError` | object has no such attribute/method (you saw this in Class 11) |

In [3]:
# Trigger each one on purpose and catch it - the message tells you exactly what broke
tests = [
    lambda: int("abc"),
    lambda: "5" + 5,
    lambda: 10 / 0,
    lambda: [1, 2][5],
    lambda: {"a": 1}["b"],
    lambda: open("does_not_exist.txt"),
]

for t in tests:
    try:
        t()
    except Exception as e:                      # 'as e' gives you the error object
        print(f"{type(e).__name__:<18} -> {e}")

ValueError         -> invalid literal for int() with base 10: 'abc'
TypeError          -> can only concatenate str (not "int") to str
ZeroDivisionError  -> division by zero
IndexError         -> list index out of range
KeyError           -> 'b'
FileNotFoundError  -> [Errno 2] No such file or directory: 'does_not_exist.txt'


## Several `except` blocks, `else` and `finally`

One `try` can have several `except` blocks — Python uses the first one that matches. Two more optional parts:

| Block | Runs when |
|-------|-----------|
| `try` | always — the risky code |
| `except X` | only if error `X` happened |
| `else` | only if **no** error happened |
| `finally` | **always**, error or not — cleanup goes here |

In [4]:
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("  Can't divide by zero")
    except TypeError:
        print("  Both values must be numbers")
    else:
        print("  Result:", result)          # only on success
    finally:
        print("  -- divide() finished --")  # always

print("divide(10, 2)");   divide(10, 2)
print("divide(10, 0)");   divide(10, 0)
print("divide(10, 'x')"); divide(10, "x")

divide(10, 2)
  Result: 5.0
  -- divide() finished --
divide(10, 0)
  Can't divide by zero
  -- divide() finished --
divide(10, 'x')
  Both values must be numbers
  -- divide() finished --


Notice `finally` printed every time. That's why it's used for cleanup — closing a file, closing a database connection, releasing a lock — things that must happen whether the work succeeded or not.

## The trap: a bare `except` hides bugs

In [ ]:
# WRONG ON PURPOSE - catches everything, including your own typo
def average(marks):
    try:
        return sum(marks) / len(mark)      # typo: 'mark' instead of 'marks'
    except:
        return 0                           # "no marks" - or is it?

print(average([80, 90, 70]))               # prints 0. The real answer is 80.

0


It printed `0` with no error — the `NameError` from the typo was silently swallowed and disguised as "no marks". That bug could sit in production for months. **Rule: always catch the specific exception you expect** (`except ZeroDivisionError:`), never a bare `except:`.

# 3. Raising your own exceptions

Sometimes Python is happy but *your* rules are broken — a negative price, an age of 300. Use `raise` to stop right there with a clear message:

In [6]:
def set_age(age):
    if not 0 <= age <= 120:
        raise ValueError(f"Age must be 0-120, got {age}")
    return age

print(set_age(25))

try:
    set_age(300)
except ValueError as e:
    print("Rejected:", e)

25
Rejected: Age must be 0-120, got 300


## Custom exceptions

For your own programs you can create an exception type — it's just a class that inherits from `Exception` (Class 11 inheritance, put to work). Callers can then catch *your* error specifically.

In [ ]:
class InsufficientFundsError(Exception):
    pass


class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(f"{self.owner} has Rs {self.balance}, tried Rs {amount}")
        self.balance -= amount


acc = Account("Ali", 5000)
try:
    acc.withdraw(8000)
except InsufficientFundsError as e:
    print("Transaction declined:", e)

Compare with Class 11, where `withdraw` just `print`ed "Refused". A print is invisible to the calling code — it can't react. An exception **forces** the caller to deal with it, and the name `InsufficientFundsError` says exactly what happened.

# 4. Files — reading and writing text

Variables vanish when the program stops. Files don't. `open(filename, mode)` gives you a file object:

| Mode | Meaning |
|------|---------|
| `"r"` | read (default) — error if file missing |
| `"w"` | write — **creates or wipes** the file |
| `"a"` | append — adds to the end |

Always use `with open(...) as f:` — it closes the file for you, even if an error happens inside (it's a `try/finally` in disguise).

In [ ]:
with open("notes.txt", "w") as f:
    f.write("Class 12: exceptions\n")
    f.write("Class 12: files\n")

with open("notes.txt", "a") as f:
    f.write("Homework: practice set\n")

with open("notes.txt") as f:
    print(f.read())

In [ ]:
# Reading line by line - the normal way for big files
with open("notes.txt") as f:
    for number, line in enumerate(f, start=1):
        print(number, line.strip())        # strip() removes the trailing \n

In [ ]:
# WRONG ON PURPOSE - "w" wipes the file before writing
with open("notes.txt", "w") as f:
    f.write("One more note\n")

with open("notes.txt") as f:
    print(f.read())                        # the first three lines are gone

No error, and three lines of data silently deleted. `"w"` means *start fresh*. If you meant to add, use `"a"`. This mistake has destroyed a lot of real log files.

In [ ]:
# Missing file = FileNotFoundError - catch it and give a useful message
try:
    with open("marks_2023.txt") as f:
        print(f.read())
except FileNotFoundError:
    print("marks_2023.txt not found - check the file name and folder")

# 5. CSV files — tables of records

CSV (comma-separated values) is how spreadsheets and most datasets travel. Python's built-in `csv` module handles the commas and quotes for you. `DictWriter` / `DictReader` let you work with each row as a dictionary.

In [ ]:
import csv

students = [
    {"name": "Ali",    "city": "Lahore",    "marks": 82},
    {"name": "Sara",   "city": "Karachi",   "marks": 91},
    {"name": "Hassan", "city": "Islamabad", "marks": 67},
]

with open("students.csv", "w", newline="") as f:     # newline="" avoids blank lines on Windows
    writer = csv.DictWriter(f, fieldnames=["name", "city", "marks"])
    writer.writeheader()
    writer.writerows(students)

with open("students.csv") as f:
    print(f.read())                                   # this is what the file really looks like

In [ ]:
# WRONG ON PURPOSE - everything read from a CSV is a string
with open("students.csv") as f:
    rows = list(csv.DictReader(f))

print(rows[0])
print("Top marks:", max(r["marks"] for r in rows))   # compares text: "91" > "82" > "67" ... looks fine?

In [ ]:
# ...until a mark has a different number of digits
rows.append({"name": "Zara", "city": "Multan", "marks": "100"})
print("Top marks:", max(r["marks"] for r in rows))   # says 91! "9" > "1" as text

CSV has no types — `"100"` is compared letter by letter with `"91"`, and `"9"` beats `"1"`. **Convert on read**: `int(r["marks"])`. And since a real CSV can contain a bad row, that conversion belongs in a `try`.

In [ ]:
with open("students.csv") as f:
    rows = list(csv.DictReader(f))

for r in rows:
    r["marks"] = int(r["marks"])

best = max(rows, key=lambda r: r["marks"])           # Class 7's key= trick
print(f"Top student: {best['name']} ({best['marks']})")

# 6. JSON files — nested data and configs

CSV is flat rows. **JSON** holds nested dictionaries and lists — exactly the shape of Python dicts. It's the format of web APIs and of every LLM API you'll call in Month 2.

| Function | What it does | Data moves from → to |
|----------|--------------|----------------------|
| `json.dump(obj, f)` | **save** a dict/list into a file | Python dict → `.json` file on disk |
| `json.load(f)` | **read** a file back into a dict/list | `.json` file on disk → Python dict |
| `json.dumps(obj)` | turn a dict into JSON **text** (no file) | Python dict → string variable |
| `json.loads(text)` | turn JSON **text** back into a dict | string variable → Python dict |

The `s` stands for *string*. Use `dump`/`load` when you're working with **files**, and `dumps`/`loads` when the JSON is just **text in a variable**, for example a reply from an API.

In [ ]:
import json

config = {
    "model": "gpt-4o-mini",
    "temperature": 0.2,
    "max_tokens": 500,
    "stop_words": ["bye", "exit"],
    "debug": False,
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)          # indent makes it human-readable

with open("config.json") as f:
    loaded = json.load(f)

print(loaded)
print(type(loaded["temperature"]), type(loaded["debug"]))   # types survive, unlike CSV

{'model': 'gpt-4o-mini', 'temperature': 0.2, 'max_tokens': 500, 'stop_words': ['bye', 'exit'], 'debug': False}
<class 'float'> <class 'bool'>


In [9]:
# An LLM API reply is JSON text - json.loads turns it into a dict you can index
reply = '{"choices": [{"message": {"role": "assistant", "content": "Helllo!"}}]}'
data = json.loads(reply)
print(data["choices"][0]["message"]["content"])

Helllo!


In [10]:
# BREAKS ON PURPOSE - JSON is stricter than Python
bad = "{'model': 'gpt-4o-mini'}"       # single quotes are fine in Python, not in JSON
json.loads(bad)

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

**JSONDecodeError: Expecting property name enclosed in double quotes.** JSON requires double quotes, lowercase `true`/`false`/`null`, and no trailing commas. When an API or an LLM returns broken JSON, this is the error you'll see — so real code wraps `json.loads` in `try/except json.JSONDecodeError`.

# 7. Real use case: load a config safely

Put it all together: load settings from a file, but fall back to defaults if the file is missing or corrupted — instead of crashing on startup.

In [ ]:
DEFAULTS = {"model": "gpt-4o-mini", "temperature": 0.7, "max_tokens": 256}

def load_config(path):
    try:
        with open(path) as f:
            user = json.load(f)
    except FileNotFoundError:
        print(f"{path} not found - using defaults")
        return DEFAULTS
    except json.JSONDecodeError as e:
        print(f"{path} is not valid JSON ({e}) - using defaults")
        return DEFAULTS
    return {**DEFAULTS, **user}             # file values override defaults

with open("broken.json", "w") as f:
    f.write("{'oops': True}")

print(load_config("config.json"))
print(load_config("missing.json"))
print(load_config("broken.json"))

# 8. Your turn

**1.** Write `safe_divide(a, b)` that raises a custom `DivisionByZeroError` (inherit from `Exception`) when `b` is `0`, and returns `a / b` otherwise. Call it inside `try/except` with `b = 0` and print a friendly message. Add a `finally` that prints `"done"`.

**2.** Ask the user for their age with `input()`. Keep asking in a `while` loop until they type a valid whole number between 1 and 120 — catch `ValueError` for non-numbers and `raise` your own `ValueError` for out-of-range numbers.

**3.** Write a list of 5 product records (`name`, `price`, `stock`) to `products.csv`. Then read it back, convert `price` and `stock` to numbers, and print the total stock value (`price × stock` summed). Add one row with `"price": "N/A"` and make your reader skip it with a warning instead of crashing.

**4.** Save a JSON config for a chatbot (`bot_name`, `greeting`, `max_turns`, `banned_words` as a list). Load it back, print the greeting, then change `max_turns`, save it again, and reload to prove the change stuck.